In [1]:
manufacturer = 'STANDARD MOTOR PRODUCTS'
type_code = '6557'

from gevent import monkey
monkey.patch_all(thread=False)
from gevent.queue import Queue
import gevent

import sys
sys.path.append('C:/Users/YangTeng/Desktop/venv/00.module')
import UA
import Proxy

# = = = = = = = = = = = = = = =

import pandas as pd

df = pd.read_excel('./blower motor resistor.xlsx',
                   header=0,
                   dtype=str).fillna('')

list_url = df['Url'].tolist()
list_part_number = df['Part_Number'].tolist()

print('总数量：' + str(len(list_url)))
print()

# = = = = = = = = = = = = = = =

df = pd.DataFrame()

总数量：671



C:\Users\YangTeng\Desktop\venv\lib\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [2]:
import requests

from bs4 import BeautifulSoup

from lxml import etree

import numpy as np

import json

# = = = = = = = = = = = = = = =

import time
from datetime import datetime

# = = = = = = = = = = = = = = =

work = Queue()
for url, part_number in zip(list_url, list_part_number):
    work.put_nowait((url, part_number))

def crawler():
    global df
    
    while not work.empty():
        url, part_number = work.get_nowait()
        
        # = = = = = = = = = = = = = = =
        
        status = 'error'
        for _ in range(30):
            try:
                           
                
                resp = requests.get(url,
                                    headers=UA.get_User_Agent_Requests(),
                                    proxies=Proxy.get_Proxy_Requests()).text
                
                # = = = = = = = = = = = = = = =
                
                soup = BeautifulSoup(resp, 'lxml')
                
                # = = = = = = = = = = = = = = =
                
                html = etree.HTML(str(soup))
                
                if len(html.xpath('span[contains(@class, "navlabellink") and contains(@class, "nvoffset") and contains(@class, "nnormal")]/text()')) != 0 and html.xpath('span[contains(@class, "navlabellink") and contains(@class, "nvoffset") and contains(@class, "nnormal")]/text()')[0] == 'No Parts Found':
                    df_temp = pd.DataFrame([{'status': 'ok',
                                             'Manufacturer': manufacturer,
                                             'Part_Number': part_number,
                                             'Url': url}])
                    
                    status = 'ok'
                    
                    break

                list_part = html.xpath('//tbody[contains(@class, "listing-inner") and (contains(@class, "altrow-a-1") or contains(@class, "altrow-a-0"))]')

                # = = = = = = = = = = = = = = =
                
                list_part_number = [_.xpath('./descendant::div[contains(@class, "listing-text-row-moreinfo-truck")]/span[contains(@title, "Buyer\'s Guide")]')[0].text for _ in list_part]
                
                # = = = = = = = = = = = = = = =
                
                number_index = np.where(np.array(list_part_number) == part_number)[0]
                
                # = = = = = = = = = = = = = = =
                
                list_manufacturer = [_.xpath('./descendant::div[contains(@class, "listing-text-row-moreinfo-truck")]/span[contains(@class, "listing-final-manufacturer")]')[0].text for _ in list_part]
                
                # = = = = = = = = = = = = = = =
                
                manufacturer_index = np.where(np.array(list_manufacturer) == manufacturer)[0]
                
                # = = = = = = = = = = = = = = =
                
                index = list(np.intersect1d(number_index, manufacturer_index))
                
                # = = = = = = = = = = = = = = =
                
                if len(index) == 1:
                    index = index[0]
                    
                    # = = = = = = = = = = = = = = =
                    
                    note_1 = list_part[index].xpath('./descendant::div[contains(@class, "listing-text-row-moreinfo-truck")]/span[contains(@class, "span-link-underline-remover")]/text()')
                    if len(note_1) == 0:
                        note_1 = ''
                    else:
                        note_1 = note_1[0].strip()
   
                    # = = = = = = = = = = = = = = =
                    
                    note_2 = list_part[index].xpath('./descendant::div[contains(@class, "listing-text-row") and not(contains(@class, "moreinfo-truck"))]/span/span/text()')
                    if len(note_2) == 0:
                        note_2 = ''
                    else:
                        note_2 = [_.strip() for _ in note_2]
                        while '' in note_2:
                            note_2.remove('')
                        note_2 = '\n'.join(note_2)

                    # = = = = = = = = = = = = = = =
                    
                    oe = list_part[index].xpath('./descendant::span[contains(@title, "Replaces these Alternate/ OE Part Numbers")]')
                    if len(oe) == 0:
                        oe = ''
                    else:
                        oe = oe[0].xpath('./text()')[0].strip().replace(', ', ';')
                        
                    # = = = = = = = = = = = = = = =
                    
                    src = list_part[index].xpath('./descendant::img[contains(@id, "inlineimg_thumb")]')
                    if len(src) == 0:
                        src = ''
                    else:
                        src = 'https://www.rockauto.com' + src[0].xpath('./@src')[0].strip().replace('__ra_m.', '.')
    
                    # = = = = = = = = = = = = = = =
                    
                    part_json = list_part[index].xpath('./descendant::input[contains(@id, "listing_data_essential")]')
                    if len(part_json) == 0:
                        part_key = ''
                    else:
                        part_key = json.loads(part_json[0].xpath('./@value')[0])['partkey'].strip()
                        
                    # = = = = = = = = = = = = = = =
                    
                    df_temp = pd.DataFrame([{'status': 'ok',
                                             '序号': '',
                                             'Manufacturer': manufacturer,
                                             'Part_Number': part_number,
                                             'Vehcile': '',
                                             'Alternate_OE_Part_Numbers': oe,
                                             'Note_1': note_1,
                                             'Note_2': note_2,
                                             'Pic': '',
                                             'Url': url,
                                             'Info': 'https://www.rockauto.com/en/moreinfo.php?pt=' + type_code + '&pk=' + part_key,
                                             'Src': src,
                                             'Type_Code': type_code,
                                             'Part_Key': part_key}])
                
                    status = 'ok'
                
                    break
                
            except:
                time.sleep(0.5)
                continue

        if status == 'error':
            df_temp = pd.DataFrame([{'status': 'error',
                                     'Manufacturer': manufacturer,
                                     'Part_Number': part_number,
                                     'Url': url}])

        df = pd.concat([df, df_temp], ignore_index=True).fillna('')

        print(part_number +  '  <->  [' + status + '] - 剩余数量：' + str(work.qsize()) + ' - 当前时间：' + datetime.now().strftime('%H:%M:%S'))


C:\Users\YangTeng\Desktop\venv\lib\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [3]:
list_task = []
for _ in range(10):
    task = gevent.spawn(crawler)
    list_task.append(task)
gevent.joinall(list_task)

df = df.drop_duplicates(ignore_index=True)

# print()
# print('爬虫输出中')
# df.to_excel('./part_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.xlsx', index=False)
# print()
# print('爬虫结束')

RU101  <->  [ok] - 剩余数量：661 - 当前时间：09:49:33
RU1004  <->  [ok] - 剩余数量：660 - 当前时间：09:49:33
RU100  <->  [ok] - 剩余数量：659 - 当前时间：09:49:33
RU1003  <->  [ok] - 剩余数量：658 - 当前时间：09:49:33
RU1012  <->  [ok] - 剩余数量：657 - 当前时间：09:49:33
RU1002  <->  [ok] - 剩余数量：656 - 当前时间：09:49:33
RU1006  <->  [ok] - 剩余数量：655 - 当前时间：09:49:33
RU1009  <->  [ok] - 剩余数量：654 - 当前时间：09:49:33
RU1014  <->  [ok] - 剩余数量：653 - 当前时间：09:49:34
RU1017  <->  [ok] - 剩余数量：652 - 当前时间：09:49:34
RU104  <->  [ok] - 剩余数量：651 - 当前时间：09:49:34
RU1023  <->  [ok] - 剩余数量：650 - 当前时间：09:49:34
RU102  <->  [ok] - 剩余数量：649 - 当前时间：09:49:34
RU103  <->  [ok] - 剩余数量：648 - 当前时间：09:49:35
RU1018  <->  [ok] - 剩余数量：647 - 当前时间：09:49:35
RU1027  <->  [ok] - 剩余数量：646 - 当前时间：09:49:35
RU1025  <->  [ok] - 剩余数量：645 - 当前时间：09:49:35
RU109  <->  [ok] - 剩余数量：644 - 当前时间：09:49:35
RU109T  <->  [ok] - 剩余数量：643 - 当前时间：09:49:36
RU207  <->  [ok] - 剩余数量：642 - 当前时间：09:49:36
RU106  <->  [ok] - 剩余数量：641 - 当前时间：09:49:36
RU1000  <->  [ok] - 剩余数量：640 - 当前时间：09:49:37
RU201  <->  [ok] -

RU399  <->  [ok] - 剩余数量：475 - 当前时间：09:50:38
RU402  <->  [ok] - 剩余数量：474 - 当前时间：09:50:39
RU401  <->  [ok] - 剩余数量：473 - 当前时间：09:50:39
RU403  <->  [ok] - 剩余数量：472 - 当前时间：09:50:39
RU395  <->  [ok] - 剩余数量：471 - 当前时间：09:50:40
RU389  <->  [ok] - 剩余数量：470 - 当前时间：09:50:40
RU403HTK  <->  [ok] - 剩余数量：469 - 当前时间：09:50:41
RU404  <->  [ok] - 剩余数量：468 - 当前时间：09:50:41
RU388  <->  [ok] - 剩余数量：467 - 当前时间：09:50:42
RU404HTK  <->  [ok] - 剩余数量：466 - 当前时间：09:50:42
RU406  <->  [ok] - 剩余数量：465 - 当前时间：09:50:42
RU407  <->  [ok] - 剩余数量：464 - 当前时间：09:50:42
RU404T  <->  [ok] - 剩余数量：463 - 当前时间：09:50:42
RU396HTK  <->  [ok] - 剩余数量：462 - 当前时间：09:50:43
RU410  <->  [ok] - 剩余数量：461 - 当前时间：09:50:43
RU405  <->  [ok] - 剩余数量：460 - 当前时间：09:50:43
RU408  <->  [ok] - 剩余数量：459 - 当前时间：09:50:43
RU412  <->  [ok] - 剩余数量：458 - 当前时间：09:50:43
RU42  <->  [ok] - 剩余数量：457 - 当前时间：09:50:44
RU415  <->  [ok] - 剩余数量：456 - 当前时间：09:50:44
RU419  <->  [ok] - 剩余数量：455 - 当前时间：09:50:45
RU429  <->  [ok] - 剩余数量：454 - 当前时间：09:50:45
RU427T  <->  [ok] - 剩余数

RU620  <->  [ok] - 剩余数量：289 - 当前时间：09:51:53
RU636  <->  [ok] - 剩余数量：288 - 当前时间：09:51:54
RU631  <->  [ok] - 剩余数量：287 - 当前时间：09:51:54
RU637  <->  [ok] - 剩余数量：286 - 当前时间：09:51:54
RU638T  <->  [ok] - 剩余数量：285 - 当前时间：09:51:54
RU64  <->  [ok] - 剩余数量：284 - 当前时间：09:51:55
RU640  <->  [ok] - 剩余数量：283 - 当前时间：09:51:55
RU644  <->  [ok] - 剩余数量：282 - 当前时间：09:51:56
RU638  <->  [ok] - 剩余数量：281 - 当前时间：09:51:56
RU646  <->  [ok] - 剩余数量：280 - 当前时间：09:51:57
RU644T  <->  [ok] - 剩余数量：279 - 当前时间：09:51:58
RU650  <->  [ok] - 剩余数量：278 - 当前时间：09:51:58
RU642  <->  [ok] - 剩余数量：277 - 当前时间：09:51:58
RU647  <->  [ok] - 剩余数量：276 - 当前时间：09:51:59
RU652  <->  [ok] - 剩余数量：275 - 当前时间：09:52:00
RU650T  <->  [ok] - 剩余数量：274 - 当前时间：09:52:00
RU651  <->  [ok] - 剩余数量：273 - 当前时间：09:52:00
RU645  <->  [ok] - 剩余数量：272 - 当前时间：09:52:00
RU648  <->  [ok] - 剩余数量：271 - 当前时间：09:52:02
RU654  <->  [ok] - 剩余数量：270 - 当前时间：09:52:02
RU650HTK  <->  [ok] - 剩余数量：269 - 当前时间：09:52:02
RU656T  <->  [ok] - 剩余数量：268 - 当前时间：09:52:04
RU653  <->  [ok] - 剩余数量：26

RU866  <->  [ok] - 剩余数量：102 - 当前时间：09:53:17
RU868  <->  [ok] - 剩余数量：101 - 当前时间：09:53:17
RU855  <->  [ok] - 剩余数量：100 - 当前时间：09:53:17
RU87  <->  [ok] - 剩余数量：99 - 当前时间：09:53:18
RU873  <->  [ok] - 剩余数量：98 - 当前时间：09:53:18
RU870  <->  [ok] - 剩余数量：97 - 当前时间：09:53:18
RU863  <->  [ok] - 剩余数量：96 - 当前时间：09:53:18
RU876  <->  [ok] - 剩余数量：95 - 当前时间：09:53:19
RU859  <->  [ok] - 剩余数量：94 - 当前时间：09:53:19
RU879  <->  [ok] - 剩余数量：93 - 当前时间：09:53:19
RU878  <->  [ok] - 剩余数量：92 - 当前时间：09:53:20
RU882  <->  [ok] - 剩余数量：91 - 当前时间：09:53:20
RU875  <->  [ok] - 剩余数量：90 - 当前时间：09:53:21
RU877  <->  [ok] - 剩余数量：89 - 当前时间：09:53:21
RU884  <->  [ok] - 剩余数量：88 - 当前时间：09:53:21
RU888  <->  [ok] - 剩余数量：87 - 当前时间：09:53:22
RU874  <->  [ok] - 剩余数量：86 - 当前时间：09:53:22
RU89  <->  [ok] - 剩余数量：85 - 当前时间：09:53:22
RU892  <->  [ok] - 剩余数量：84 - 当前时间：09:53:23
RU890  <->  [ok] - 剩余数量：83 - 当前时间：09:53:23
RU883  <->  [ok] - 剩余数量：82 - 当前时间：09:53:23
RU894  <->  [ok] - 剩余数量：81 - 当前时间：09:53:24
RU893  <->  [ok] - 剩余数量：80 - 当前时间：09:53:24
RU895  <->

C:\Users\YangTeng\Desktop\venv\lib\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [4]:
df.to_excel('C:/Users/YangTeng/Desktop/文档/爬虫/blower motor resistor-detail-RockAuto.xlsx', index=False)
print()
print('搞定')


搞定
